# **Tahap 4**

In [17]:
print("Nama-nama kolom yang Anda miliki di dataset:")
print(df_train.columns.tolist())

Nama-nama kolom yang Anda miliki di dataset:
['Nomor Perkara', 'Tanggal Putusan', 'Jenis Perkara', 'Pasal', 'Pihak Penggugat/Pemohon', 'Pihak Tergugat/Termohon/Terdakwa', 'Dakwaan', 'Barang Bukti']


In [18]:
kolom_solusi = 'Pasal'
case_solutions = dict(zip(df_train['Nomor Perkara'], df_train[kolom_solusi]))

def predict_outcome(query: str, k: int = 5) -> str:
    query_vektor = vectorizer.transform([str(query)])
    skor_kemiripan = cosine_similarity(query_vektor, X_train)[0]

    top_k_indices = skor_kemiripan.argsort()[-k:][::-1]

    skor_kandidat_solusi = {}
    for idx in top_k_indices:
        case_id = df_train['Nomor Perkara'].values[idx]
        bobot = skor_kemiripan[idx]
        solusi = str(case_solutions[case_id])

        if solusi in skor_kandidat_solusi:
            skor_kandidat_solusi[solusi] += bobot
        else:
            skor_kandidat_solusi[solusi] = bobot

    predicted_solution = max(skor_kandidat_solusi, key=skor_kandidat_solusi.get)
    return predicted_solution

print("Sel 7 Selesai Fungsi predict_outcome() dan kamus solusi berhasil dibuat!")

Sel 7 Selesai Fungsi predict_outcome() dan kamus solusi berhasil dibuat!


In [19]:

print("=== DEMO MANUAL PREDIKSI PASAL (CASE REUSE) ===\n")

for i in range(5):
    kasus_baru_id = df_test.iloc[i]['Nomor Perkara']
    teks_kasus_baru = df_test.iloc[i]['Dakwaan']
    solusi_asli = str(df_test.iloc[i]['Pasal'])

    prediksi_mesin = predict_outcome(query=teks_kasus_baru, k=5)

    print(f"Kasus Uji {i+1} : {kasus_baru_id}")
    print(f"Pasal Asli    : {solusi_asli}")
    print(f"Tebakan Mesin : {prediksi_mesin}")


    list_asli = set([p.strip() for p in solusi_asli.split(',')])
    list_tebakan = set([p.strip() for p in prediksi_mesin.split(',')])

    pasal_cocok = list_asli.intersection(list_tebakan)

    if len(pasal_cocok) > 0:
        print(f"Status        : PREDIKSI BENAR (Berhasil menebak {len(pasal_cocok)} pasal yang sama)")
        print(f"Pasal Cocok   : {', '.join(pasal_cocok)}")
    else:
        print("Status        : PREDIKSI SALAH TOTAL")
    print("-" * 50)

=== DEMO MANUAL PREDIKSI PASAL (CASE REUSE) ===

Kasus Uji 1 : 27-k/pm.i-02/ad/ill/2025
Pasal Asli    : pasal 26, pasal 2, pasal 8, pasal 55 ayat (1), pasal 7, pasal 173 ayat (6), pasal 55 ayat (1) ke-1, pasal 5, pasal 175 ayat (1), pasal 180 ayat (1), pasal 176, pasal 171, pasal 173 ayat (1), pasal 155 ayat (1), pasal 190 ayat (1), pasal 44, pasal 372, pasal 172 ayat (1), pasal 378, pasal 139, pasal 1365
Tebakan Mesin : pasal 143, pasal 58, pasal 5 ayat (1), pasal 124 ayat (4), pasal 155 ayat (1), pasal 85, pasal 26, pasal 2 ayat (4), pasal 46 ayat (1), pasal 141 ayat (10), pasal 1, pasal 86, pasal 139, pasal 190 ayat (1), pasal 44, pasal 87 ayat (1) ke-2
Status        : PREDIKSI BENAR (Berhasil menebak 5 pasal yang sama)
Pasal Cocok   : pasal 44, pasal 155 ayat (1), pasal 190 ayat (1), pasal 26, pasal 139
--------------------------------------------------
Kasus Uji 2 : 91-k/pm
Pasal Asli    : pasal 143, pasal 58, pasal 124 ayat (4), pasal 155 ayat (1), pasal 85, pasal 26, pasal 46 ay

In [20]:
import os
import pandas as pd

print("=== MEMPROSES DAN MENYIMPAN FILE PREDIKSI (predictions.csv) ===")

data_output = []

for i in range(len(df_test)):
    query_id = df_test.iloc[i]['Nomor Perkara']
    teks_kasus_baru = df_test.iloc[i]['Dakwaan']

    predicted_solution = predict_outcome(query=teks_kasus_baru, k=5)

    query_vektor = vectorizer.transform([str(teks_kasus_baru)])
    skor_kemiripan = cosine_similarity(query_vektor, X_train)[0]
    top_5_indices = skor_kemiripan.argsort()[-5:][::-1]

    list_top_5 = [df_train['Nomor Perkara'].values[idx] for idx in top_5_indices]
    top_5_case_ids_str = ", ".join(list_top_5)

    data_output.append({
        "query_id": query_id,
        "predicted_solution": predicted_solution,
        "top_5_case_ids": top_5_case_ids_str
    })

df_predictions = pd.DataFrame(data_output)

folder_path = os.path.join("data", "results")
file_path = os.path.join(folder_path, "predictions.csv")

# Buat folder 'data/results' jika belum ada
os.makedirs(folder_path, exist_ok=True)

# Simpan ke CSV tanpa nomor indeks bawaan
#df_predictions.to_csv(file_path, index=False)

#print(f"\nSUKSES! File CSV telah berhasil dibuat di: {file_path}")
print("Berikut adalah 5 baris pertama isinya:")
df_predictions.head()

=== MEMPROSES DAN MENYIMPAN FILE PREDIKSI (predictions.csv) ===
Berikut adalah 5 baris pertama isinya:


,query_id,predicted_solution,top_5_case_ids
0,27-k/pm.i-02/ad/ill/2025,"pasal 143, pasal 58, pasal 5 ayat (1), pasal 1...","36-k/pm.i-02/au/iv/2025, 20-k/pm.1-02/ad/11/20..."
1,91-k/pm,"pasal 143, pasal 58, pasal 5 ayat (1), pasal 1...","36-k/pm.i-02/au/iv/2025, 67-k/pm.i-02/ad/viii/..."
2,55-k/pm.i-02/ad/vi/2021,pasal 288 ayat (2),"6-p/pm.ii-09/ad/vi/2025, 74, 20-k/pm.1-02/ad/1..."
3,52-k/pm.ii-09/ad/ili/2025,"pasal 45, pasal 58, pasal 143, pasal 46, pasal...","62-k/pm.ii-09/au/lv/2025, 29-k/pmt.i/bdg/ad/iv..."
4,363,"pasal 143, pasal 58, pasal 5 ayat (1), pasal 1...","36-k/pm.i-02/au/iv/2025, 20-k/pm.1-02/ad/11/20..."
